# Modulo 7 · Lezione 7 — Cross-Validation e GridSearchCV

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Perché serve la CV: il problema del test set unico |
| 2 | K-Fold Cross-Validation e varianti |
| 3 | `cross_val_score` |
| 4 | GridSearchCV — ricerca esaustiva |
| 5 | RandomizedSearchCV — ricerca efficiente |
| 6 | Pipeline + GridSearchCV |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.metrics import accuracy_score

np.random.seed(42)
print("Librerie pronte")

---
## 1. Il problema del test set unico

```
Valutazione ingenua:
  Dividere i dati → allenare → valutare sul test → riportare lo score
  Problema: lo score dipende da QUALI dati sono finiti nel test.
  Con pochi campioni, la varianza dello score può essere molto alta.

Esempio:
  100 campioni, test_size=0.2 → 20 campioni di test
  Un modello può avere accuracy 0.75 o 0.90 solo per caso
  a seconda di quali 20 campioni sono finiti nel test.

Soluzione: Cross-Validation
  Usa TUTTI i dati sia per allenare che per validare,
  a turni diversi. Dà uno score più affidabile e una stima
  della sua incertezza (deviazione standard).
```

In [ ]:
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
print(f"Dataset: {X.shape[0]} campioni, {X.shape[1]} feature")

# Dimostrare la variabilità dello score con un unico test set
modelo = DecisionTreeClassifier(max_depth=4, random_state=None)  # senza seed fissa

scores_aleatorios = []
for i in range(30):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2)
    scores_aleatorios.append(accuracy_score(y_te, modelo.fit(X_tr, y_tr).predict(X_te)))

print(f"\n30 valutazioni con test set casuale:")
print(f"  Minimo: {min(scores_aleatorios):.4f}")
print(f"  Massimo: {max(scores_aleatorios):.4f}")
print(f"  Intervallo:  {max(scores_aleatorios)-min(scores_aleatorios):.4f}  ← molto variabile!")

plt.figure(figsize=(8, 3))
plt.plot(range(30), scores_aleatorios, 'bo-', ms=5, alpha=0.7)
plt.axhline(np.mean(scores_aleatorios), color='red', lw=2, ls='--', label=f'Media={np.mean(scores_aleatorios):.4f}')
plt.fill_between(range(30), np.mean(scores_aleatorios)-np.std(scores_aleatorios),
                 np.mean(scores_aleatorios)+np.std(scores_aleatorios), alpha=0.2, color='red')
plt.xlabel('Esperimento'); plt.ylabel('Accuracy')
plt.title('Variabilità dello score con test set casuale')
plt.legend(); plt.tight_layout(); plt.show()

---
## 2. K-Fold Cross-Validation

```
Algoritmo K-Fold (K=5 tipico):

  Dati completi: [████████████████████] (100 campioni)

  Fold 1: [Test][Train][Train][Train][Train]
  Fold 2: [Train][Test][Train][Train][Train]
  Fold 3: [Train][Train][Test][Train][Train]
  Fold 4: [Train][Train][Train][Test][Train]
  Fold 5: [Train][Train][Train][Train][Test]

  → 5 modelli allenati, 5 score
  → score_finale = media(5 score) ± std(5 score)

Varianti:
  StratifiedKFold  → mantiene la proporzione delle classi in ogni fold
                     (usare SEMPRE in classificazione)
  LeaveOneOut      → K = n_campioni (molto costoso, dataset piccoli)
  RepeatedKFold    → ripete K-Fold più volte con seed diverse
```

In [ ]:
# Visualizzare i fold
X_demo = np.arange(10).reshape(-1, 1)
y_demo = np.array([0,1,0,1,0,1,0,1,0,1])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (nombre, cv) in zip(axes, [
    ('KFold (K=5)', KFold(n_splits=5, shuffle=True, random_state=42)),
    ('StratifiedKFold (K=5)', StratifiedKFold(n_splits=5, shuffle=True, random_state=42)),
]):
    grid = np.zeros((5, 10))
    for fold_idx, (tr_idx, te_idx) in enumerate(cv.split(X_demo, y_demo)):
        grid[fold_idx, tr_idx] = 1
        grid[fold_idx, te_idx] = 2

    ax.imshow(grid, cmap=plt.cm.get_cmap('RdYlGn', 3), aspect='auto', vmin=0, vmax=2)
    ax.set_xticks(range(10))
    ax.set_xticklabels([f's{i}\n(y={y_demo[i]})' for i in range(10)], fontsize=8)
    ax.set_yticks(range(5))
    ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
    ax.set_title(nombre)

from matplotlib.patches import Patch
legend = [Patch(facecolor='red', label='Test'), Patch(facecolor='green', label='Train')]
axes[0].legend(handles=legend, loc='lower right', fontsize=8)
plt.suptitle('Struttura dei fold (verde=train, rosso=test)', fontsize=10)
plt.tight_layout(); plt.show()

---
## 3. `cross_val_score`

In [ ]:
X, y = load_breast_cancer(return_X_y=True)

modelos = [
    ('Albero depth=1', DecisionTreeClassifier(max_depth=1, random_state=42)),
    ('Albero depth=5', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('Random Forest', RandomForestClassifier(100, random_state=42, n_jobs=-1)),
    ('LogisticReg',   LogisticRegression(max_iter=5000, random_state=42)),
]

print(f"{'Modello':20} {'Acc CV (media±std)':22} {'Min':8} {'Max':8}")
print("-" * 65)

resultados = {}
for nombre, m in modelos:
    scores = cross_val_score(
        m, X, y,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='accuracy',
        n_jobs=-1
    )
    resultados[nombre] = scores
    print(f"{nombre:20} {scores.mean():.4f} ± {scores.std():.4f}         "
          f"{scores.min():.4f}  {scores.max():.4f}")

# Visualizzare
fig, ax = plt.subplots(figsize=(9, 4))
nombres = list(resultados.keys())
datos   = list(resultados.values())
ax.boxplot(datos, labels=nombres, patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
ax.set_ylabel('Accuracy')
ax.set_title('Distribuzione degli score — 5-Fold CV')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 4. GridSearchCV — ricerca esaustiva di iperparametri

```
Problema: quali valori di max_depth e n_estimators funzionano meglio?

Senza GridSearch: provare manualmente combinazione per combinazione.
Con GridSearch: definire la griglia, sklearn prova TUTTE le combinazioni
                e seleziona la migliore secondo la CV.

Esempio: max_depth=[3,5,10] × n_estimators=[50,100]
  → 6 combinazioni × 5 fold = 30 allenamenti

Risultato: .best_params_  → combinazione migliore
           .best_score_   → score CV con quella combinazione
           .best_estimator_ → modello già allenato su tutti i dati
```

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                            stratify=y, random_state=42)

# Definire la griglia di iperparametri
rejilla = {
    'max_depth':     [3, 5, 8, None],
    'n_estimators':  [50, 100, 200],
    'min_samples_split': [2, 5, 10],
}

gs = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=rejilla,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy',
    refit=True,     # riallena con la miglior combinazione su tutti i dati di train
    n_jobs=-1,
    verbose=1,
)
gs.fit(X_tr, y_tr)

print(f"\nMigliori iperparametri:")
for k, v in gs.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nScore CV (train): {gs.best_score_:.4f}")
print(f"Accuracy sul test: {accuracy_score(y_te, gs.predict(X_te)):.4f}")

# Visualizzare i risultati come heatmap (n_estimators vs max_depth)
resultados_df = pd.DataFrame(gs.cv_results_)

# Filtrare a min_samples_split=2 per visualizzare in 2D
sub = resultados_df[resultados_df['param_min_samples_split'] == 2].copy()
sub['max_depth_str'] = sub['param_max_depth'].apply(lambda x: str(x))

pivot = sub.pivot_table(
    values='mean_test_score',
    index='max_depth_str',
    columns='param_n_estimators'
)

fig, ax = plt.subplots(figsize=(7, 4))
import matplotlib
im = ax.imshow(pivot.values, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('n_estimators'); ax.set_ylabel('max_depth')
ax.set_title('Accuracy CV (min_samples_split=2)')
plt.colorbar(im, ax=ax)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.3f}', ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.show()

---
## 5. RandomizedSearchCV — ricerca efficiente

```
Problema di GridSearch: cresce esponenzialmente con più iperparametri.
  3 param × 4 valori ciascuno = 64 combinazioni
  5 param × 4 valori = 1024 combinazioni

RandomizedSearchCV: campiona n_iter combinazioni a caso.
  → Molto più veloce.
  → Quasi altrettanto valido in pratica (paper Bergstra & Bengio 2012).
  → Permette intervalli continui (scipy.stats.distributions).

Quando usare ciascuno:
  GridSearch:    ≤3 iperparametri, intervalli piccoli e discreti
  RandomSearch:  molti iperparametri, intervalli continui, poco tempo
```

In [ ]:
from scipy.stats import randint, uniform
import time

# Spazio di ricerca con distribuzioni continue
espacio = {
    'n_estimators':       randint(50, 500),
    'max_depth':          [3, 5, 8, 10, None],
    'min_samples_split':  randint(2, 15),
    'min_samples_leaf':   randint(1, 8),
    'max_features':       ['sqrt', 'log2', 0.5, 0.7],
}

rs = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=espacio,
    n_iter=40,    # provare 40 combinazioni a caso
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy',
    refit=True,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

t0 = time.time()
rs.fit(X_tr, y_tr)
t1 = time.time()

print(f"RandomizedSearchCV completato in {t1-t0:.1f}s")
print(f"\nMigliori iperparametri:")
for k, v in rs.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nScore CV (train): {rs.best_score_:.4f}")
print(f"Accuracy sul test: {accuracy_score(y_te, rs.predict(X_te)):.4f}")

# Confronto rapido
print(f"\n{'Metodo':25} {'Score CV':>10} {'Acc test':>10}")
print("-" * 48)
print(f"{'GridSearchCV':25} {gs.best_score_:>10.4f} {accuracy_score(y_te, gs.predict(X_te)):>10.4f}")
print(f"{'RandomizedSearchCV':25} {rs.best_score_:>10.4f} {accuracy_score(y_te, rs.predict(X_te)):>10.4f}")

---
## 6. Pipeline + GridSearchCV

```
Il pattern più importante nel ML di produzione:
  Pipeline(steps) + GridSearchCV → ricerca senza data leakage

Sintassi per nominare gli iperparametri in Pipeline:
  'nomePasso__iperparametro'

  Esempio:
    Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())])
    param_grid = {'clf__C': [0.01, 0.1, 1, 10], 'clf__penalty': ['l1', 'l2']}

  Lo scaler si adatta sul train di ogni fold e trasforma il test → SENZA leakage.
```

In [ ]:
# Pipeline con LogisticRegression
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=5000, random_state=42)),
])

rejilla_pipe = {
    'clf__C':       [0.001, 0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l1', 'l2'],
    'clf__solver':  ['liblinear'],  # supporta l1 e l2
}

gs_pipe = GridSearchCV(
    pipe, rejilla_pipe,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy',
    refit=True, n_jobs=-1
)
gs_pipe.fit(X_tr, y_tr)

print("Pipeline + GridSearchCV — Logistic Regression")
print(f"Migliori parametri: {gs_pipe.best_params_}")
print(f"Score CV:           {gs_pipe.best_score_:.4f}")
print(f"Accuracy test:      {accuracy_score(y_te, gs_pipe.predict(X_te)):.4f}")

# Visualizzare: C vs penalty
df_res = pd.DataFrame(gs_pipe.cv_results_)
pivot2 = df_res.pivot_table(
    values='mean_test_score',
    index='param_clf__penalty',
    columns='param_clf__C'
)

fig, ax = plt.subplots(figsize=(9, 3))
im2 = ax.imshow(pivot2.values, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(pivot2.columns)))
ax.set_xticklabels(pivot2.columns)
ax.set_yticks(range(len(pivot2.index)))
ax.set_yticklabels(pivot2.index)
ax.set_xlabel('C'); ax.set_ylabel('penalty')
ax.set_title('Accuracy CV — Pipeline LogisticRegression')
plt.colorbar(im2, ax=ax)
for i in range(len(pivot2.index)):
    for j in range(len(pivot2.columns)):
        ax.text(j, i, f'{pivot2.values[i,j]:.3f}', ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.show()

---
## Pratica

### Esercizio 1 — Confrontare CV vs holdout

**Consegna:** Genera un dataset piccolo:

```python
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=150, n_features=10, n_informative=5, random_state=42)
```

1. Valuta un DecisionTree depth=4 con 50 split casuali (test_size=0.2)
2. Valuta lo stesso albero con 5-Fold StratifiedCV, 10 ripetizioni (usa `RepeatedStratifiedKFold`)
3. Quale stima ha meno varianza? Perché?
4. Di quale metodo ti fidi di più per scegliere un modello?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score, RepeatedStratifiedKFold
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=150, n_features=10, n_informative=5, random_state=42)
modelo = DecisionTreeClassifier(max_depth=4)

# 1. Holdout casuale
scores_holdout = []
for _ in range(50):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2)
    scores_holdout.append(accuracy_score(y_te, modelo.fit(X_tr, y_tr).predict(X_te)))

# 2. Repeated Stratified CV
rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scores_cv = cross_val_score(modelo, X, y, cv=rkf, scoring='accuracy', n_jobs=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(scores_holdout, bins=15, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(np.mean(scores_holdout), color='red', lw=2, label=f'Media={np.mean(scores_holdout):.4f}')
axes[0].set_title(f'Holdout casuale (n=50)\nStd={np.std(scores_holdout):.4f}')
axes[0].legend()

axes[1].hist(scores_cv, bins=15, color='green', alpha=0.7, edgecolor='white')
axes[1].axvline(np.mean(scores_cv), color='red', lw=2, label=f'Media={np.mean(scores_cv):.4f}')
axes[1].set_title(f'Repeated StratifiedKFold (5×10)\nStd={np.std(scores_cv):.4f}')
axes[1].legend()

plt.tight_layout(); plt.show()

print(f"Holdout:  media={np.mean(scores_holdout):.4f}, std={np.std(scores_holdout):.4f}")
print(f"CV:       media={np.mean(scores_cv):.4f}, std={np.std(scores_cv):.4f}")
print(f"\nLa CV riduce la std. Fidarsi di più della CV per selezionare modelli.")
```
</details>

---
### Esercizio 2 — GridSearch per un albero di decisione

**Consegna:** Usa il dataset del cancro al seno:

1. Definisci un `GridSearchCV` per `DecisionTreeClassifier` esplorando:
   - `max_depth`: [2, 3, 4, 5, 6, 8, None]
   - `min_samples_split`: [2, 5, 10, 20]
   - `criterion`: ['gini', 'entropy']
2. Mostra i migliori parametri e lo score sul test
3. Confronta l'albero di default (non ottimizzato) vs quello ottimizzato
4. Visualizza la heatmap degli score per criterion=gini

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rejilla = {
    'max_depth':         [2, 3, 4, 5, 6, 8, None],
    'min_samples_split': [2, 5, 10, 20],
    'criterion':         ['gini', 'entropy'],
}

gs = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    rejilla,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy', refit=True, n_jobs=-1
)
gs.fit(X_tr, y_tr)

print(f"Migliori parametri: {gs.best_params_}")
print(f"Score CV: {gs.best_score_:.4f}")

# Confronto
arbol_default = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)
print(f"\n{'Modello':25} {'Acc test':>10}")
print("-" * 38)
print(f"{'Albero di default':25} {accuracy_score(y_te, arbol_default.predict(X_te)):>10.4f}")
print(f"{'Albero ottimizzato':25} {accuracy_score(y_te, gs.predict(X_te)):>10.4f}")

# Heatmap
df_res = pd.DataFrame(gs.cv_results_)
sub = df_res[df_res['param_criterion'] == 'gini'].copy()
sub['depth_str'] = sub['param_max_depth'].apply(lambda x: str(x))
pivot = sub.pivot_table('mean_test_score', 'depth_str', 'param_min_samples_split')

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel('min_samples_split'); ax.set_ylabel('max_depth')
ax.set_title('Accuracy CV (criterion=gini)')
plt.colorbar(im, ax=ax)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.3f}', ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 3 — Pipeline completo con ricerca di iperparametri

**Consegna:** Crea una pipeline che includa scaling + Random Forest e cerca i suoi migliori iperparametri:

```python
from sklearn.datasets import load_wine
X, y = load_wine(return_X_y=True)
```

1. Pipeline: `StandardScaler → RandomForestClassifier`
2. Usa `RandomizedSearchCV` per esplorare:
   - `clf__n_estimators`: randint(50, 300)
   - `clf__max_depth`: [3, 5, None]
   - `clf__max_features`: ['sqrt', 'log2']
   - `clf__min_samples_leaf`: randint(1, 6)
3. n_iter=30, cv=5
4. Mostra i migliori parametri, lo score CV e l'accuracy sul test

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
from scipy.stats import randint
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = load_wine(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(random_state=42, n_jobs=-1)),
])

espacio = {
    'clf__n_estimators':   randint(50, 300),
    'clf__max_depth':      [3, 5, None],
    'clf__max_features':   ['sqrt', 'log2'],
    'clf__min_samples_leaf': randint(1, 6),
}

rs = RandomizedSearchCV(
    pipe, espacio, n_iter=30,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy', refit=True, n_jobs=-1, random_state=42
)
rs.fit(X_tr, y_tr)

print("Migliori iperparametri:")
for k, v in rs.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nScore CV:      {rs.best_score_:.4f}")
print(f"Accuracy test: {accuracy_score(y_te, rs.predict(X_te)):.4f}")
```
</details>

---
### Esercizio 4 — Metriche di valutazione multiple

**Consegna:** Vogliamo valutare un classificatore non solo per accuracy ma anche per F1:

```python
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=500, n_features=15, n_informative=8,
                           weights=[0.7, 0.3], random_state=42)  # classi sbilanciate
```

1. Valuta LogisticRegression con `cross_validate` usando accuracy, F1 e AUC
2. Quale metrica cambia di più tra i fold?
3. La tua decisione tra accuracy e F1 cambierebbe in questo caso?

*Indizio: usa `from sklearn.model_selection import cross_validate`*

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=500, n_features=15, n_informative=8,
                           weights=[0.7, 0.3], random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
m  = LogisticRegression(max_iter=5000, random_state=42)

resultados = cross_validate(
    m, X, y, cv=cv,
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
    return_train_score=False, n_jobs=-1
)

print(f"{'Metrica':12} {'Media':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 48)
for k in ['accuracy', 'f1', 'roc_auc']:
    sc = resultados[f'test_{k}']
    print(f"{k:12} {sc.mean():>8.4f} {sc.std():>8.4f} {sc.min():>8.4f} {sc.max():>8.4f}")

print("\nConclusione:")
print("  Con classi sbilanciate (70/30), l'accuracy può essere ingannevole.")
print("  F1 e AUC danno un'immagine più onesta delle prestazioni reali.")
print("  Usa F1 quando i falsi negativi e i falsi positivi hanno un costo.")
```
</details>

---
### Esercizio 5 (sfida) — Comparatore automatico di modelli

**Consegna:** Implementa `comparar_modelos(modelos_dict, X, y, cv=5)` che:

1. Valuta ogni modello con StratifiedKFold e calcola accuracy, F1 e AUC
2. Restituisce un `pd.DataFrame` ordinato per AUC decrescente
3. Traccia un grafico a barre con error bar (std) per le 3 metriche
4. Evidenzia il modello migliore nel grafico

Provala con: LogisticRegression, DecisionTree depth=5, RandomForest 100.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

def comparar_modelos(modelos_dict, X, y, cv=5):
    cv_obj = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    metricas = {'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'}
    filas = []
    for nombre, m in modelos_dict.items():
        res = cross_validate(m, X, y, cv=cv_obj, scoring=metricas, n_jobs=-1)
        fila = {'modelo': nombre}
        for metrica in metricas:
            sc = res[f'test_{metrica}']
            fila[f'{metrica}_media'] = sc.mean()
            fila[f'{metrica}_std']   = sc.std()
        filas.append(fila)

    df = pd.DataFrame(filas).sort_values('roc_auc_media', ascending=False)

    # Grafico
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    colores = ['steelblue', 'green', 'coral']
    mejor = df.iloc[0]['modelo']

    for ax, met, color in zip(axes, ['accuracy', 'f1', 'roc_auc'], colores):
        medias = df[f'{met}_media'].values
        stds   = df[f'{met}_std'].values
        nombres = df['modelo'].values
        barras = ax.bar(nombres, medias, yerr=stds, capsize=5,
                        color=[color if n != mejor else 'gold' for n in nombres],
                        edgecolor='gray')
        ax.set_ylim(0.7, 1.05)
        ax.set_title(met.upper())
        ax.tick_params(axis='x', rotation=15)
        ax.grid(True, axis='y', alpha=0.3)
        for barra, media, std in zip(barras, medias, stds):
            ax.text(barra.get_x() + barra.get_width()/2, media + std + 0.005,
                    f'{media:.3f}', ha='center', va='bottom', fontsize=9)

    plt.suptitle(f'Confronto tra modelli — Migliore: {mejor}', fontsize=11)
    plt.tight_layout(); plt.show()
    return df

# Prova
X, y = load_breast_cancer(return_X_y=True)
modelos = {
    'LogisticReg':   LogisticRegression(max_iter=5000, random_state=42),
    'Albero depth=5': DecisionTreeClassifier(max_depth=5, random_state=42),
    'RandomForest': RandomForestClassifier(100, random_state=42, n_jobs=-1),
}
tabla = comparar_modelos(modelos, X, y, cv=5)
print(tabla[['modelo','accuracy_media','f1_media','roc_auc_media']].to_string(index=False))
```
</details>

---
## Riepilogo

```python
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline

# Valutazione affidabile
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(modelo, X, y, cv=cv, scoring='accuracy')
print(f"{scores.mean():.4f} ± {scores.std():.4f}")

# Ricerca di iperparametri (esaustiva)
gs = GridSearchCV(modelo, param_grid={'max_depth': [3,5,None]}, cv=cv, refit=True)
gs.fit(X_tr, y_tr)
print(gs.best_params_, gs.best_score_)

# Ricerca di iperparametri (efficiente)
rs = RandomizedSearchCV(modelo, param_distributions=espacio, n_iter=40, cv=cv, refit=True)

# Pipeline + GridSearch (senza data leakage)
pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())])
gs_pipe = GridSearchCV(pipe, {'clf__C': [0.1, 1, 10]}, cv=cv)
gs_pipe.fit(X_tr, y_tr)
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 8** impareremo il **Clustering**: KMeans, il metodo Elbow e il Silhouette score per trovare gruppi in dati senza etichette.